# 07 Prophet Regressors Forecast

Prophet に `baseline_m4` の外生変数を追加し、fixed B の conditional forecast として評価する。既存の Prophet baseline は holidaysなし・regressorsなしの unconditional forecast だったが、この notebook では SSM/SARIMAX と同じ外生ダミーを既知として test 期間に与える。

これは fixed B のみを対象とする。fixed A では COVID系ダミーや `post_stat_change` が学習期間で変動しないため、まず fixed B で比較する。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.evaluation import evaluate_forecasts
from src.forecasting.prophet_model import (
    fit_prophet_with_regressors,
    forecast_prophet_with_regressors,
    prepare_prophet_regressors_from_spec,
    select_nonconstant_regressors,
)
from src.forecasting.splits import make_fixed_split_b


DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
PREDICTIONS_DIR = FORECAST_DIR / "predictions"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

for path in [PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SPEC_NAME = "baseline_m4"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Forecast output:", FORECAST_DIR)

Project root: C:\Users\fugat\Desktop\python_project\Transport_amount_project
Data path: C:\Users\fugat\Desktop\python_project\Transport_amount_project\data\processed\parcel_volume_connected.csv
Forecast output: C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts


## 1. データ読み込みと fixed B split

接続済みデータを読み込み、fixed B を作成する。評価は `number_parcels` スケールで行うが、Prophet の学習は `y = log(number_parcels)` で行う。

In [2]:
df = load_connected_parcel_data(str(DATA_PATH))
split = make_fixed_split_b(df)
train = split["train"]
test = split["test"]

split_summary = pd.DataFrame(
    [
        {
            "split": split["split"],
            "train_start": train.index.min().date(),
            "train_end": train.index.max().date(),
            "train_rows": len(train),
            "test_start": test.index.min().date(),
            "test_end": test.index.max().date(),
            "test_rows": len(test),
        }
    ]
)
display(split_summary)

,split,train_start,train_end,train_rows,test_start,test_end,test_rows
0,fixed_b,2002-04-01,2023-12-01,261,2024-01-01,2026-02-01,26


## 2. baseline_m4 regressors の作成

`baseline_m4` の外生変数を train/test 全体で作成し、学習期間で定数の列を除外する。train/test の列順は揃える。

In [3]:
combined = pd.concat([train, test])
regressors_all = prepare_prophet_regressors_from_spec(combined, SPEC_NAME)
regressors_train = regressors_all.loc[train.index]
regressors_test = regressors_all.loc[test.index]
regressors_train_selected, regressors_test_selected, used_regressors = select_nonconstant_regressors(
    regressors_train,
    regressors_test,
)

regressor_summary = pd.DataFrame(
    [
        {
            "regressor": col,
            "train_active_months": int(regressors_train[col].sum()),
            "test_active_months": int(regressors_test[col].sum()),
            "used": col in used_regressors,
        }
        for col in regressors_all.columns
    ]
)

print("Used regressors:", used_regressors)
display(regressor_summary)

Used regressors: ['hike_dummy', 'covid_main', 'covid_wave1', 'covid_2021', 'post_stat_change']


,regressor,train_active_months,test_active_months,used
0,hike_dummy,27,0,True
1,covid_main,39,0,True
2,covid_wave1,2,0,True
3,covid_2021,9,0,True
4,post_stat_change,17,26,True


## 3. Prophet + regressors の学習と予測

Prophet の設定は baseline と同様に、holidaysなし、weekly/daily seasonalityなし、yearly seasonalityありとする。ここでは test 期間の外生ダミーを既知として与えるため、`forecast_type` は conditional である。

In [4]:
model = fit_prophet_with_regressors(
    train,
    regressors_train_selected,
    target_col="y",
    yearly_seasonality=True,
)

forecast_df = forecast_prophet_with_regressors(
    model,
    test,
    regressors_test_selected,
    split="fixed_b",
    spec_name=SPEC_NAME,
)

display(forecast_df.head())
display(forecast_df.tail())

16:31:00 - cmdstanpy - INFO - Chain [1] start processing


16:31:00 - cmdstanpy - INFO - Chain [1] done processing


,date,cutoff,horizon,y_true,y_pred,model,split,forecast_type,spec_name,target_col,target_scale
0,2024-01-01,2023-12-01,1,348749.0,330175.548695,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original
1,2024-02-01,2023-12-01,2,344158.0,339449.498313,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original
2,2024-03-01,2023-12-01,3,392055.0,370944.884223,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original
3,2024-04-01,2023-12-01,4,369921.0,371531.903898,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original
4,2024-05-01,2023-12-01,5,369040.0,357596.653227,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original


,date,cutoff,horizon,y_true,y_pred,model,split,forecast_type,spec_name,target_col,target_scale
21,2025-10-01,2023-12-01,22,405217.0,388957.802982,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original
22,2025-11-01,2023-12-01,23,409319.0,417109.477578,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original
23,2025-12-01,2023-12-01,24,510426.0,559783.055952,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original
24,2026-01-01,2023-12-01,25,380997.0,347521.935445,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original
25,2026-02-01,2023-12-01,26,349832.0,352908.886358,prophet_regressors,fixed_b,conditional,baseline_m4,number_parcels,original


## 4. 評価と既存モデルとの比較

評価は `number_parcels` スケールで行う。既存の fixed B metrics があれば、Prophet baseline、SSM conditional、SARIMAX grid best と比較する。

In [5]:
prophet_regressors_metrics = evaluate_forecasts(
    forecast_df,
    y_train=train["number_parcels"],
)
display(prophet_regressors_metrics)

comparison_frames = []
for metrics_name in [
    "prophet_metrics.csv",
    "ssm_metrics.csv",
    "sarimax_grid_best_models.csv",
]:
    metrics_path = METRICS_DIR / metrics_name
    if metrics_path.exists():
        metrics_df = pd.read_csv(metrics_path)
        metrics_df["source_file"] = metrics_name
        comparison_frames.append(metrics_df)

comparison_rows = []
for frame in comparison_frames:
    for _, row in frame[frame["split"] == "fixed_b"].iterrows():
        model_name = row["model"]
        if row.get("source_file") == "sarimax_grid_best_models.csv":
            model_name = f"{model_name}_grid_best"
        comparison_rows.append(
            {
                "model": model_name,
                "source_file": row.get("source_file", ""),
                "rmse": row["rmse"],
                "mae": row["mae"],
                "mape": row["mape"],
                "mase": row["mase"],
            }
        )

comparison_rows.append(
    {
        "model": "prophet_regressors",
        "source_file": "prophet_regressors_metrics.csv",
        "rmse": prophet_regressors_metrics.loc[0, "rmse"],
        "mae": prophet_regressors_metrics.loc[0, "mae"],
        "mape": prophet_regressors_metrics.loc[0, "mape"],
        "mase": prophet_regressors_metrics.loc[0, "mase"],
    }
)

comparison = pd.DataFrame(comparison_rows).sort_values("rmse")
display(comparison)

,model,split,forecast_type,spec_name,n,rmse,mae,mape,mase
0,prophet_regressors,fixed_b,conditional,baseline_m4,26,17877.060334,13052.309116,3.201742,1.043615


,model,source_file,rmse,mae,mape,mase
3,sarimax_grid_best,sarimax_grid_best_models.csv,7975.062046,6701.292189,1.709270,0.535811
1,ssm,ssm_metrics.csv,8112.663937,6756.339867,1.718500,0.540212
2,sarima_grid_best,sarimax_grid_best_models.csv,14790.762992,12236.132613,3.071175,0.978357
4,prophet_regressors,prophet_regressors_metrics.csv,17877.060334,13052.309116,3.201742,1.043615
0,prophet,prophet_metrics.csv,26730.003403,19845.066371,4.834269,1.586739


## 5. 保存

既存の Prophet baseline 出力は上書きせず、Prophet regressors 用の別ファイルとして保存する。

In [6]:
forecast_df.to_csv(PREDICTIONS_DIR / "fixed_b_prophet_regressors.csv", index=False)
prophet_regressors_metrics.to_csv(METRICS_DIR / "prophet_regressors_metrics.csv", index=False)

fit_summary = pd.DataFrame(
    [
        {
            "model": "prophet_regressors",
            "split": "fixed_b",
            "forecast_type": "conditional",
            "information_set": "conditional_exog_known",
            "spec_name": SPEC_NAME,
            "used_regressors": ",".join(used_regressors),
            "yearly_seasonality": True,
            "weekly_seasonality": False,
            "daily_seasonality": False,
            "holidays": False,
            "uncertainty_samples": 0,
        }
    ]
)
fit_summary.to_csv(METRICS_DIR / "prophet_regressors_fit_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(10.5, 5.5))
train_tail = train.tail(24)
ax.plot(train_tail.index, train_tail["number_parcels"], color="0.55", linewidth=1.2, label="train actual tail")
ax.plot(test.index, test["number_parcels"], color="black", linewidth=1.6, label="test actual")
ax.plot(forecast_df["date"], forecast_df["y_pred"], marker="o", linewidth=1.2, label="prophet_regressors")
ax.axvline(train.index.max(), color="0.2", linestyle=":", linewidth=1.0)
ax.set_title("fixed_b: Prophet regressors forecast comparison")
ax.set_xlabel("Date")
ax.set_ylabel("number_parcels")
ax.grid(True, color="0.85", linewidth=0.8)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fixed_b_prophet_regressors_forecast.png", dpi=300, bbox_inches="tight")
plt.close(fig)

display(fit_summary)
print("Saved Prophet regressors outputs.")

,model,split,forecast_type,information_set,spec_name,used_regressors,yearly_seasonality,weekly_seasonality,daily_seasonality,holidays,uncertainty_samples
0,prophet_regressors,fixed_b,conditional,conditional_exog_known,baseline_m4,"hike_dummy,covid_main,covid_wave1,covid_2021,p...",True,False,False,False,0


Saved Prophet regressors outputs.


## 6. 読み取りメモ

Prophet regressors は conditional forecast である。test期間の `baseline_m4` 外生ダミーを既知として与えているため、外生変数なしの Prophet baseline とは情報条件が異なる。

比較では、Prophet without regressors から改善したか、SSM conditional や SARIMAX grid best と比べてどの程度かを確認する。